<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW12 · Session 2 · Scatter, Stacked, and Box Plots for Business Stories

        **Course level:** developing beginner  
        **Prior learning:** matplotlib fundamentals and clean sales data from TW12 Session 1  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        More complex charts help compare relationships, composition, and distributions. The goal remains a defensible business message rather than visual decoration.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - create and interpret scatter plots
- use colour and transparency without obscuring evidence
- build stacked charts for totals and proportions
- compare distributions with box plots
- assemble a focused multi-panel story and identify misleading design


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Prepare plotting data

This notebook repeats essential setup so it can run independently.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

sales = pd.read_csv(DATA_DIR / "business_sales_dirty.csv")
sales["region"] = sales["region"].astype("string").str.strip().str.title().fillna("Unknown")
sales["channel"] = sales["channel"].astype("string").str.strip().str.title().replace({"Telephone": "Phone"})
sales["order_date"] = pd.to_datetime(sales["order_date"], format="mixed", dayfirst=True, errors="coerce")
sales["units"] = sales["units"].mask(~sales["units"].between(1, 20))
sales["unit_price"] = sales["unit_price"].mask(sales["unit_price"] <= 0)
sales = sales.drop_duplicates().dropna(subset=["order_date", "units", "unit_price"]).copy()
sales["net_revenue"] = sales["units"] * sales["unit_price"] * (1 - sales["discount"])
print(sales.shape)


## 1. Scatter plots show paired numerical relationships

Each point represents one observation. Position encodes two numerical values. Look for direction, form, strength, clusters, changing spread, and unusual observations before calculating a coefficient.


### Plot marketing spend against revenue

Transparency reveals overlapping points. A small amount of jitter is not used because both variables are already continuous.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    sales["marketing_spend"],
    sales["net_revenue"],
    alpha=0.6,
    color="#2563eb",
    edgecolor="none",
)
ax.set_title("Marketing spend and transaction revenue")
ax.set_xlabel("Marketing spend (£)")
ax.set_ylabel("Net revenue (£)")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)


### Add a fitted line and correlation

The line summarises a linear pattern; it does not prove that changing marketing caused revenue to change.


In [ ]:
plot_data = sales[["marketing_spend", "net_revenue"]].dropna()
slope, intercept = np.polyfit(plot_data["marketing_spend"], plot_data["net_revenue"], 1)
x_line = np.linspace(plot_data["marketing_spend"].min(), plot_data["marketing_spend"].max(), 100)
r_value = stats.pearsonr(plot_data["marketing_spend"], plot_data["net_revenue"]).statistic

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(plot_data["marketing_spend"], plot_data["net_revenue"], alpha=0.55)
ax.plot(x_line, slope * x_line + intercept, color="#dc2626", linewidth=2)
ax.set_title(f"Linear association: r = {r_value:.2f}")
ax.set_xlabel("Marketing spend (£)")
ax.set_ylabel("Net revenue (£)")
fig.tight_layout()
plt.show()
plt.close(fig)


### Use colour for a small number of groups

A stable colour map and legend make channel differences inspectable.


In [ ]:
channel_colours = {"Online": "#2563eb", "Store": "#d97706", "Phone": "#059669"}
fig, ax = plt.subplots(figsize=(7, 5))
for channel, group in sales.groupby("channel"):
    ax.scatter(
        group["units"],
        group["net_revenue"],
        label=channel,
        color=channel_colours.get(channel, "#6b7280"),
        alpha=0.6,
    )
ax.set_title("Revenue versus units by channel")
ax.set_xlabel("Units")
ax.set_ylabel("Net revenue (£)")
ax.legend(title="Channel")
fig.tight_layout()
plt.show()
plt.close(fig)


## 2. Stacked charts show composition

Absolute stacks compare both total size and composition. A 100% stacked chart compares proportions but hides total size. Choose according to the question and avoid too many segments.


### Create an absolute stacked bar chart

The pivot table makes the data structure explicit before plotting.


In [ ]:
channel_region_revenue = pd.pivot_table(
    sales,
    index="region",
    columns="channel",
    values="net_revenue",
    aggfunc="sum",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(8, 5))
channel_region_revenue.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_title("Regional revenue composition by channel")
ax.set_xlabel("Region")
ax.set_ylabel("Net revenue (£)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Channel")
fig.tight_layout()
plt.show()
plt.close(fig)


### Convert totals to proportions

Dividing each row by its row total produces a 100% composition view.


In [ ]:
channel_mix = channel_region_revenue.div(channel_region_revenue.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
channel_mix.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_title("Channel share within each region")
ax.set_xlabel("Region")
ax.set_ylabel("Share of regional revenue")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. Box plots compare distributions

A box plot shows the median, quartiles, whiskers, and individually plotted potential outliers. It does not show every detail or prove that flagged points are errors.


### Compare transaction revenue across channels

The same y-axis supports a distribution comparison.


In [ ]:
channel_order = ["Online", "Store", "Phone"]
box_values = [
    sales.loc[sales["channel"] == channel, "net_revenue"].dropna()
    for channel in channel_order
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(box_values, labels=channel_order, patch_artist=True)
ax.set_title("Transaction revenue distribution by channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Net revenue (£)")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)


### Support the plot with numerical summaries

A chart and table together reduce ambiguity.


In [ ]:
channel_distribution = sales.groupby("channel")["net_revenue"].agg(
    count="count",
    median="median",
    mean="mean",
    q1=lambda values: values.quantile(0.25),
    q3=lambda values: values.quantile(0.75),
)
print(channel_distribution.round(2))


## 4. Avoid misleading visual stories

Check truncated axes, unequal interval spacing, hidden missing periods, dual axes, cherry-picked ranges, excessive smoothing, inappropriate 3D, and colours that imply unsupported meaning. A chart can be numerically correct yet rhetorically misleading.


### Demonstrate a truncated-axis effect

Both panels use identical data. The narrow y-axis on the right visually exaggerates a small difference.


In [ ]:
values = pd.Series({"A": 96, "B": 100})
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(values.index, values.values, color="#2563eb")
axes[0].set_ylim(0, 105)
axes[0].set_title("Zero baseline")
axes[1].bar(values.index, values.values, color="#dc2626")
axes[1].set_ylim(94, 101)
axes[1].set_title("Truncated baseline")
fig.suptitle("Identical values, very different impression")
fig.tight_layout()
plt.show()
plt.close(fig)


## Guided story: a focused two-panel dashboard

The first panel answers where revenue comes from; the second shows how individual transaction values vary by channel. The panels support one coherent message rather than displaying every available metric.


### Build the dashboard

Consistent labels and a restrained palette help the audience compare evidence.


In [ ]:
revenue_by_region = sales.groupby("region")["net_revenue"].sum().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].barh(revenue_by_region.index, revenue_by_region.values, color="#2563eb")
axes[0].set_title("Total revenue by region")
axes[0].set_xlabel("Net revenue (£)")

axes[1].boxplot(box_values, labels=channel_order, patch_artist=True)
axes[1].set_title("Transaction value by channel")
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Net revenue (£)")

fig.suptitle("Revenue scale and transaction distribution")
fig.tight_layout()
plt.show()
plt.close(fig)


## Student coding lab

Every chart must include a one-sentence interpretation and one limitation as comments.


        ### Student task 1: Create a scatter plot

        <div class="task"><strong>Your job:</strong> Plot unit_price against net_revenue, use transparency, calculate Pearson r, and include it in the title.</div>

        **Check your work**

        - Correlation uses paired complete values.
- Axes include units.
- The interpretation avoids causal language.


In [ ]:
# STUDENT TASK 1
scatter_data = sales[["unit_price", "net_revenue"]].dropna()
r_value = 0.0
fig, ax = plt.subplots(figsize=(7, 5))
# Add points, title, and labels.
plt.close(fig)
print("r:", r_value)


        ### Student task 2: Compare groups with colour

        <div class="task"><strong>Your job:</strong> Plot marketing_spend against net_revenue with one colour per category and a readable legend.</div>

        **Check your work**

        - Every category appears once in the legend.
- Transparency handles overlap.
- A limitation notes possible confounding or outliers.


In [ ]:
# STUDENT TASK 2
fig, ax = plt.subplots(figsize=(7, 5))
# Loop over category groups and draw each group.
plt.close(fig)
print(sales["category"].value_counts())


        ### Student task 3: Build a stacked total chart

        <div class="task"><strong>Your job:</strong> Create a region-by-category revenue pivot and draw an absolute stacked bar chart.</div>

        **Check your work**

        - The pivot uses sum and fills missing combinations with zero.
- The y-axis reports pounds.
- The interpretation discusses both totals and composition.


In [ ]:
# STUDENT TASK 3
region_category = pd.DataFrame()
fig, ax = plt.subplots(figsize=(8, 5))
# Create the pivot and stacked plot.
plt.close(fig)
print(region_category)


        ### Student task 4: Create a 100% composition chart

        <div class="task"><strong>Your job:</strong> Convert the previous pivot to row proportions, verify each non-empty row sums to 1, and plot it.</div>

        **Check your work**

        - Every row is normalised separately.
- The axis runs from 0 to 1.
- The limitation says total revenue size is no longer visible.


In [ ]:
# STUDENT TASK 4
region_category_share = pd.DataFrame()
fig, ax = plt.subplots(figsize=(8, 5))
# Calculate shares, assert row sums, and plot.
plt.close(fig)
print(region_category_share)


        ### Student task 5: Compare distributions

        <div class="task"><strong>Your job:</strong> Create box plots of satisfaction by channel and accompany them with count, median, and IQR per channel.</div>

        **Check your work**

        - Missing satisfaction is excluded explicitly.
- Channels use a shared scale.
- The numerical table supports the visual comparison.



**Optional extension:** Overlay lightly jittered individual observations while preserving anonymity.


In [ ]:
# STUDENT TASK 5
satisfaction_summary = pd.DataFrame()
fig, ax = plt.subplots(figsize=(7, 5))
# Build both the plot and summary.
plt.close(fig)
print(satisfaction_summary)


        ## Knowledge check

        Answer these without running new code first.

        1. What does one scatter-plot point represent?
2. What is lost in a 100% stacked chart?
3. What information does a box plot omit?
4. Why can a truncated bar axis mislead?
5. What makes several panels a coherent dashboard?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW13 combines import, profiling, cleaning, feature engineering, and visual EDA in one end-to-end case study.
